# Pitcher Deep Dive Analysis

**Purpose:** Advanced analysis of pitcher WAR predictions

**Prerequisites:** Must run pitcher_pipeline_main.ipynb first

**Last Updated:** 2025-10-06

---

## Analysis Overview
1. Elite pitcher performance (>5 WAR/162)
2. Replacement level analysis (<0 WAR/162)
3. Feature correlation heatmap
4. Partial dependence plots (K%, BB%, damage_control_ratio)
5. SHAP values for top 20 pitchers
6. Error analysis by year and team
7. Prediction interval analysis
8. Model component comparison
9. Outlier investigation (>2 sigma residuals)

In [ ]:
# Cell 1: Imports and Load Saved Data

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import os
from datetime import datetime

# Add project root to path
project_root = Path('.').absolute().parent.parent.parent
sys.path.insert(0, str(project_root))

# Load most recent predictions
predictions_dir = project_root / 'predictions'
pred_files = [f for f in os.listdir(predictions_dir) if 'pitcher' in f and f.endswith('.csv')]
latest_pred_file = sorted(pred_files)[-1]
df_predictions = pd.read_csv(predictions_dir / latest_pred_file)

print(f"Loaded predictions: {latest_pred_file}")
print(f"  Total predictions: {len(df_predictions)}")
if 'Role' in df_predictions.columns:
    print(f"  Roles: {df_predictions['Role'].value_counts().to_dict()}")

# Load saved model
models_dir = project_root / 'models'
model_files = [f for f in os.listdir(models_dir) if 'pitcher' in f and f.endswith('.pkl')]
latest_model_file = sorted(model_files)[-1]
pitcher_model = joblib.load(models_dir / latest_model_file)

print(f"\nLoaded model: {latest_model_file}")
print("Ready for deep-dive analysis!")

In [ ]:
# Cell 2: Elite Pitcher Analysis (>5 WAR/162)

# Calculate residuals if not already in predictions
if 'Residual' not in df_predictions.columns and 'Actual_WAR_per_162' in df_predictions.columns:
    df_predictions['Residual'] = df_predictions['Actual_WAR_per_162'] - df_predictions['Predicted_WAR_per_162']

# Filter elite pitchers
elite_threshold = 5.0

if 'Actual_WAR_per_162' in df_predictions.columns:
    elite_pitchers = df_predictions[df_predictions['Actual_WAR_per_162'] > elite_threshold]
    
    print("=" * 60)
    print(f"ELITE PITCHERS (>{elite_threshold} WAR/162)")
    print("=" * 60)
    print(f"Count: {len(elite_pitchers)}")
    print(f"Mean Actual WAR: {elite_pitchers['Actual_WAR_per_162'].mean():.2f}")
    print(f"Mean Predicted WAR: {elite_pitchers['Predicted_WAR_per_162'].mean():.2f}")
    print(f"Mean Residual: {elite_pitchers['Residual'].mean():.2f}")
    print(f"MAE: {elite_pitchers['Residual'].abs().mean():.2f}")
    
    # Show biggest under-predictions
    print("\nTop 5 Under-predicted Elite Pitchers:")
    under_predicted = elite_pitchers.nlargest(5, 'Residual')
    display_cols = ['Name', 'IP', 'Actual_WAR_per_162', 'Predicted_WAR_per_162', 'Residual']
    if 'Role' in under_predicted.columns:
        display_cols.insert(1, 'Role')
    print(under_predicted[display_cols].to_string(index=False))
else:
    print("Note: Actual WAR data not available in predictions (current season projections only)")

In [ ]:
# Cell 3: Replacement Level Analysis (<0 WAR/162)

if 'Actual_WAR_per_162' in df_predictions.columns:
    # Filter replacement level pitchers
    replacement = df_predictions[df_predictions['Actual_WAR_per_162'] < 0]
    
    print("=" * 60)
    print("REPLACEMENT LEVEL PITCHERS (<0 WAR/162)")
    print("=" * 60)
    print(f"Count: {len(replacement)}")
    print(f"Mean Actual WAR: {replacement['Actual_WAR_per_162'].mean():.2f}")
    print(f"Mean Predicted WAR: {replacement['Predicted_WAR_per_162'].mean():.2f}")
    print(f"Mean Residual: {replacement['Residual'].mean():.2f}")
    print(f"MAE: {replacement['Residual'].abs().mean():.2f}")
    
    # Show biggest over-predictions
    print("\nTop 5 Over-predicted Replacement Pitchers:")
    over_predicted = replacement.nsmallest(5, 'Residual')
    display_cols = ['Name', 'IP', 'Actual_WAR_per_162', 'Predicted_WAR_per_162', 'Residual']
    if 'Role' in over_predicted.columns:
        display_cols.insert(1, 'Role')
    print(over_predicted[display_cols].to_string(index=False))
else:
    print("Note: Actual WAR data not available (current season projections only)")

In [ ]:
# Cell 4: Feature Correlation Heatmap

from new_pipeline.notebooks.shared.plotting_utils import create_correlation_heatmap
from new_pipeline.notebooks.shared.pipeline_runner import load_historical_data
from new_pipeline.common.constants import PITCHER_MODEL_FEATURES

# Load historical data for correlation analysis
print("Loading historical data for correlation analysis...")
pitcher_historical = load_historical_data(player_type='pitcher', years=[2024])

# Add WAR for correlation
feature_cols_with_war = PITCHER_MODEL_FEATURES + ['WAR']
available_cols = [c for c in feature_cols_with_war if c in pitcher_historical.columns]

fig = create_correlation_heatmap(pitcher_historical, features=available_cols)
fig.update_layout(title="Pitcher Feature Correlation Matrix (2024)")
fig.show()

print("\nAnalysis Questions:")
print("- Which features are highly correlated? (e.g., K% and strikeout_efficiency)")
print("- Which features are independent predictors?")
print("- How do composite features correlate with WAR?")

In [ ]:
# Cell 5: Partial Dependence Plot - K%

from new_pipeline.notebooks.shared.plotting_utils import create_partial_dependence
from new_pipeline.notebooks.shared.pipeline_runner import run_data_pipeline

# Process historical data through pipeline
print("Processing data for partial dependence analysis...")
pitcher_processed = run_data_pipeline(pitcher_historical, player_type='pitcher')

# Filter starters only for K% analysis
pitcher_processed['GS_per_G'] = pitcher_processed['GS'] / pitcher_processed['G'].replace(0, 1)
starters = pitcher_processed[pitcher_processed['GS_per_G'] > 0.7]

X_starters = starters[PITCHER_MODEL_FEATURES].values

# Get starter model from ensemble
if hasattr(pitcher_model, 'models') and 'starter' in pitcher_model.models:
    starter_model = pitcher_model.models['starter']
else:
    starter_model = pitcher_model  # Use unified model if no role split

fig_k = create_partial_dependence(starter_model, X_starters, feature='K%', feature_names=PITCHER_MODEL_FEATURES)
fig_k.update_layout(title="Partial Dependence: K% (Starters)")
fig_k.show()

print("\nExpected Pattern: Linear positive relationship (higher K% -> higher WAR)")

In [ ]:
# Cell 6: Partial Dependence Plot - BB%

fig_bb = create_partial_dependence(starter_model, X_starters, feature='BB%', feature_names=PITCHER_MODEL_FEATURES)
fig_bb.update_layout(title="Partial Dependence: BB% (Starters)")
fig_bb.show()

print("\nExpected Pattern: Linear negative relationship (higher BB% -> lower WAR)")

In [ ]:
# Cell 7: Partial Dependence Plot - damage_control_ratio

fig_dcr = create_partial_dependence(starter_model, X_starters, feature='damage_control_ratio', feature_names=PITCHER_MODEL_FEATURES)
fig_dcr.update_layout(title="Partial Dependence: damage_control_ratio (Starters)")
fig_dcr.show()

print("\nExpected Pattern: Non-linear (optimal range, diminishing returns at extremes)")

In [ ]:
# Cell 8: SHAP Values - Top 20 Pitchers

from new_pipeline.notebooks.shared.analysis_utils import calculate_shap_values
import shap
import matplotlib.pyplot as plt

# Calculate SHAP for top 20 pitchers by WAR
if 'WAR' in pitcher_processed.columns:
    top_20 = pitcher_processed.nlargest(20, 'WAR')
    X_top_20 = top_20[PITCHER_MODEL_FEATURES].values
    
    print("Calculating SHAP values for top 20 pitchers...")
    print("This may take 2-5 minutes...\n")
    
    shap_values = calculate_shap_values(starter_model, X_top_20, background_samples=100)
    
    # Waterfall plot for #1 pitcher
    print(f"SHAP Analysis: {top_20.iloc[0]['Name']}")
    shap.plots.waterfall(shap_values[0], show=False)
    plt.title(f"SHAP Analysis: {top_20.iloc[0]['Name']}")
    plt.show()
    
    # Summary plot for all top 20
    shap.plots.beeswarm(shap_values, show=False)
    plt.title("Feature Impact on Top 20 Pitchers")
    plt.show()
    
    print("\nAnalysis Questions:")
    print("- Which features drive elite predictions?")
    print("- Are any features surprisingly low impact?")
else:
    print("WAR data not available for SHAP analysis")

In [ ]:
# Cell 9: Error Analysis by Year

import plotly.graph_objects as go

# Load predictions for multiple years (if available)
years_available = [2022, 2023, 2024]
error_by_year = {}

for year in years_available:
    try:
        year_files = [f for f in os.listdir(predictions_dir) 
                     if 'pitcher' in f and str(year) in f and f.endswith('.csv')]
        if year_files:
            df_year = pd.read_csv(predictions_dir / year_files[0])
            if 'Residual' in df_year.columns:
                residuals = df_year['Residual'].values
                error_by_year[year] = {
                    'MAE': np.abs(residuals).mean(),
                    'RMSE': np.sqrt((residuals**2).mean()),
                    'Mean Error': residuals.mean()
                }
    except Exception as e:
        continue

if error_by_year:
    # Plot year-over-year performance
    years_list = list(error_by_year.keys())
    mae_list = [error_by_year[y]['MAE'] for y in years_list]
    
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=years_list, y=mae_list, mode='lines+markers',
                             name='MAE', line=dict(width=3)))
    fig.update_layout(title="Model Performance Over Time",
                     xaxis_title="Year", yaxis_title="MAE")
    fig.show()
    
    print("Year-by-year MAE:")
    for year in years_list:
        print(f"  {year}: {error_by_year[year]['MAE']:.3f}")
else:
    print("Multi-year prediction data not available yet")

In [ ]:
# Cell 10: Error Analysis by Team

if 'Team' in df_predictions.columns and 'Residual' in df_predictions.columns:
    # Analyze if certain teams have systematically better/worse predictions
    team_errors = df_predictions.groupby('Team')['Residual'].agg(['mean', 'std', 'count'])
    team_errors = team_errors[team_errors['count'] >= 5]  # Teams with 5+ pitchers
    
    # Top 5 over-predicted teams
    print("=" * 60)
    print("TEAMS WITH LARGEST OVER-PREDICTIONS")
    print("=" * 60)
    print(team_errors.nsmallest(5, 'mean').to_string())
    
    # Top 5 under-predicted teams
    print("\n" + "=" * 60)
    print("TEAMS WITH LARGEST UNDER-PREDICTIONS")
    print("=" * 60)
    print(team_errors.nlargest(5, 'mean').to_string())
    
    print("\nNote: Could indicate park factor issues or team-specific strategies")
else:
    print("Team data or residuals not available for analysis")

In [ ]:
# Cell 11: Prediction Intervals

# Analyze prediction uncertainty using quantile predictions from ensemble
print("Analyzing prediction intervals...\n")

for role_name in ['starter', 'reliever', 'swing']:
    if hasattr(pitcher_model, 'models') and role_name in pitcher_model.models:
        role_model = pitcher_model.models[role_name]
        
        # Get role-specific data
        if role_name == 'starter':
            role_data = pitcher_processed[pitcher_processed['GS_per_G'] > 0.7]
        elif role_name == 'reliever':
            role_data = pitcher_processed[pitcher_processed['GS_per_G'] < 0.1]
        else:
            role_data = pitcher_processed[(pitcher_processed['GS_per_G'] >= 0.1) & 
                                         (pitcher_processed['GS_per_G'] <= 0.7)]
        
        if len(role_data) > 0:
            X_role = role_data[PITCHER_MODEL_FEATURES].values
            y_role = role_data['WAR_per_162'].values
            
            # Extract quantile predictions if available
            if hasattr(role_model, 'predict_quantiles'):
                q50 = role_model.predict_quantiles(X_role, quantile=0.50)
                q25 = role_model.predict_quantiles(X_role, quantile=0.25)
                q75 = role_model.predict_quantiles(X_role, quantile=0.75)
                
                print(f"=" * 50)
                print(f"{role_name.upper()} Prediction Intervals")
                print(f"=" * 50)
                print(f"Median width (q75-q25): {np.median(q75 - q25):.2f} WAR")
                coverage = ((y_role >= q25) & (y_role <= q75)).mean()
                print(f"Coverage (% within interval): {coverage:.1%}\n")
            else:
                print(f"{role_name.upper()}: Quantile predictions not available\n")

if not hasattr(pitcher_model, 'models'):
    print("Unified model - role-based analysis not applicable")

In [ ]:
# Cell 12: Model Component Comparison

from new_pipeline.notebooks.shared.analysis_utils import compare_models
import plotly.express as px

# Extract base models from ensemble
if hasattr(pitcher_model, 'models'):
    starter_ensemble = pitcher_model.models['starter']
else:
    starter_ensemble = pitcher_model

base_models = {}
if hasattr(starter_ensemble, 'rf_model'):
    base_models['RandomForest'] = starter_ensemble.rf_model
if hasattr(starter_ensemble, 'keras_model'):
    base_models['Keras'] = starter_ensemble.keras_model
if hasattr(starter_ensemble, 'xgb_model'):
    base_models['XGBoost'] = starter_ensemble.xgb_model

if len(base_models) > 0:
    # Compare on starter test set
    X_test = X_starters
    y_test = starters['WAR_per_162'].values
    
    comparison = compare_models(base_models, X_test, y_test)
    
    print("=" * 60)
    print("MODEL COMPONENT COMPARISON")
    print("=" * 60)
    print(comparison.to_string(index=False))
    
    # Visualize
    fig = px.bar(comparison, x='Model', y=['MAE', 'RMSE'],
                title="Ensemble Component Performance",
                barmode='group')
    fig.show()
else:
    print("Base models not accessible for comparison")

In [ ]:
# Cell 13: Outlier Investigation

from new_pipeline.notebooks.shared.analysis_utils import find_outliers

if 'Residual' in df_predictions.columns:
    # Find outliers (>2 std dev)
    outlier_mask = find_outliers(df_predictions['Residual'].values, threshold=2.0)
    outliers = df_predictions[outlier_mask]
    
    print("=" * 60)
    print("OUTLIERS (>2 sigma residuals)")
    print("=" * 60)
    print(f"Count: {len(outliers)} ({len(outliers)/len(df_predictions):.1%})")
    
    # Investigate top outliers
    print("\nLargest Residuals:")
    display_cols = ['Name', 'IP']
    if 'Role' in outliers.columns:
        display_cols.append('Role')
    if 'Actual_WAR_per_162' in outliers.columns:
        display_cols.extend(['Actual_WAR_per_162', 'Predicted_WAR_per_162', 'Residual'])
    
    print(outliers.nlargest(10, 'Residual')[display_cols].to_string(index=False))
    
    # Common characteristics?
    if 'Role' in outliers.columns:
        print("\nOutlier Role Distribution:")
        print(outliers['Role'].value_counts().to_string())
    
    print("\nAnalysis Questions:")
    print("- Are outliers concentrated in certain roles?")
    print("- Are they mostly low-IP pitchers (small sample noise)?")
    print("- Are there specific teams or parks?")
else:
    print("Residual data not available for outlier analysis")

print("\n" + "=" * 60)
print("PITCHER DEEP DIVE COMPLETE")
print("=" * 60)